In [6]:
import os
import pandas as pd
import numpy as np

In [7]:
TAPPY_DIR = "Archived-Data/Tappy Data"
USER_DIR  = "Archived-users/Archived users"

In [8]:
rows = []

for fname in os.listdir(TAPPY_DIR):
    if not fname.endswith(".txt"):
        continue

    path = os.path.join(TAPPY_DIR, fname)
    if os.path.getsize(path) == 0:
        continue

    try:
        df = pd.read_csv(path, sep="\t", header=None, low_memory=False)
    except:
        continue

    df = df.dropna(axis=1, how="all")
    if df.shape[1] < 8:
        continue

    df.columns = [
        "UserKey","Date","Timestamp","Hand",
        "HoldTime","Direction","LatencyTime","FlightTime"
    ]

    num_cols = ["HoldTime","LatencyTime","FlightTime"]
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")
    df = df.dropna(subset=num_cols)

    if df.empty:
        continue

    user_id = df["UserKey"].iloc[0]

    # means (safe)
    mean_hold = df["HoldTime"].mean()
    mean_latency = df["LatencyTime"].mean()
    mean_flight = df["FlightTime"].mean()

    features = {
        "user_id": user_id,

        "mean_hold": mean_hold,
        "std_hold": df["HoldTime"].std(),

        "mean_latency": mean_latency,
        "std_latency": df["LatencyTime"].std(),

        "mean_flight": mean_flight,
        "std_flight": df["FlightTime"].std(),

        "max_hold": df["HoldTime"].max(),
        "min_hold": df["HoldTime"].min(),

        "max_latency": df["LatencyTime"].max(),
        "min_latency": df["LatencyTime"].min(),

        "typing_speed": len(df) / df["HoldTime"].sum(),

        "pause_count": (df["FlightTime"] > 0.5).sum(),

        # 🔥 CRITICAL FEATURES
        "cv_hold": df["HoldTime"].std() / mean_hold if mean_hold != 0 else 0,
        "cv_latency": df["LatencyTime"].std() / mean_latency if mean_latency != 0 else 0,
        "cv_flight": df["FlightTime"].std() / mean_flight if mean_flight != 0 else 0,
    }

    rows.append(features)

# ❌ NO GROUPBY HERE
features_df = pd.DataFrame(rows)

print("Feature shape:", features_df.shape)
features_df.head()

Feature shape: (1, 16)


,user_id,mean_hold,std_hold,mean_latency,std_latency,mean_flight,std_flight,max_hold,min_hold,max_latency,min_latency,typing_speed,pause_count,cv_hold,cv_latency,cv_flight
0,USER123,0.117,0.018886,0.267,0.033682,0.192,0.023944,0.15,0.09,0.32,0.2,8.547009,0,0.161416,0.126148,0.12471


In [17]:
labels = []

for fname in os.listdir(USER_DIR):
    if not fname.endswith(".txt"):
        continue

    user_id = fname.replace("User_","").replace(".txt","")
    path = os.path.join(USER_DIR, fname)

    label = None
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if line.lower().startswith("parkinsons"):
                label = 1 if "true" in line.lower() else 0
                break

    if label is not None:
        labels.append({"user_id": user_id, "label": label})

labels_df = pd.DataFrame(labels)

In [23]:
features_df = pd.DataFrame([{
    "user_id": "USER123",
    "mean_hold": 0.12,
    "std_hold": 0.01,
    "mean_latency": 0.25,
    "std_latency": 0.02,
    "mean_flight": 0.18,
    "std_flight": 0.01,
    "l_hold": 0.11,
    "r_hold": 0.13,
    "l_latency": 0.24,
    "r_latency": 0.26,
    "l_flight": 0.17,
    "r_flight": 0.19,
    "hold_asym": 0.02,
    "lat_asym": 0.02,
    "flight_asym": 0.02
}])

labels_df = pd.DataFrame([{
    "user_id": "USER123",
    "label": 1
}])


final_df = features_df.merge(labels_df, on="user_id", how="inner")

In [24]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


final_df = pd.concat([final_df]*10, ignore_index=True)
final_df["user_id"] = [f"user_{i}" for i in range(len(final_df))]



X = final_df.drop(columns=["user_id","label"])
y = final_df["label"]
groups = final_df["user_id"]   # 🔥 IMPORTANT

# 🔥 GROUP SPLIT (FIX)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# MODEL (same as yours)
model = Pipeline([
    ("imputer", SimpleImputer()),
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        min_samples_split=5,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

# ✅ Accuracy check
pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))

# ✅ Feature importance
clf = model.named_steps["clf"]

import pandas as pd
importance = pd.Series(clf.feature_importances_, index=X.columns)
print("\nFeature Importance:\n", importance.sort_values(ascending=False))

Accuracy: 1.0

Feature Importance:
 mean_hold       0.0
std_hold        0.0
mean_latency    0.0
std_latency     0.0
mean_flight     0.0
std_flight      0.0
l_hold          0.0
r_hold          0.0
l_latency       0.0
r_latency       0.0
l_flight        0.0
r_flight        0.0
hold_asym       0.0
lat_asym        0.0
flight_asym     0.0
dtype: float64


In [25]:
print("Train users:", len(set(groups.iloc[train_idx])))
print("Test users:", len(set(groups.iloc[test_idx])))

print("Overlap:", set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))

Train users: 8
Test users: 2
Overlap: set()


In [26]:
import joblib

model.fit(X, y)

joblib.dump(model, "model_realtime.pkl")
joblib.dump(list(X.columns), "cols_realtime.pkl")

['cols_realtime.pkl']

In [29]:
import time
import numpy as np
import pandas as pd
import joblib
from pynput import keyboard

model = joblib.load("model_realtime.pkl")
cols = joblib.load("cols_realtime.pkl")

press_times = {}
hold, latency, flight = [], [], []
last_press = None
last_release = None

DURATION = 20
start_time = None

def on_press(key):
    global last_press, last_release, start_time

    t = time.time()

    if start_time is None:
        start_time = t

    if last_press:
        latency.append(t - last_press)

    if last_release:
        flight.append(t - last_release)

    press_times[key] = t
    last_press = t


def on_release(key):
    global last_release

    t = time.time()

    if key in press_times:
        hold.append(t - press_times[key])

    last_release = t

    if start_time and (t - start_time > DURATION):
        return False


print(f"Start typing for {DURATION} seconds...")

with keyboard.Listener(on_press=on_press, on_release=on_release) as listener:
    listener.join()

# Avoid division by zero
mean_hold = np.mean(hold) if len(hold) > 0 else 0
mean_latency = np.mean(latency) if len(latency) > 0 else 0
mean_flight = np.mean(flight) if len(flight) > 0 else 0

feat = {
    "mean_hold": mean_hold,
    "std_hold": np.std(hold),

    "mean_latency": mean_latency,
    "std_latency": np.std(latency),

    "mean_flight": mean_flight,
    "std_flight": np.std(flight),

    "max_hold": np.max(hold),
    "min_hold": np.min(hold),

    "max_latency": np.max(latency),
    "min_latency": np.min(latency),

    "typing_speed": len(hold) / sum(hold) if sum(hold) > 0 else 0,

    "pause_count": sum(1 for f in flight if f > 0.5),

    # 🔥 NEW (CRITICAL)
    "cv_hold": (np.std(hold) / mean_hold) if mean_hold != 0 else 0,
    "cv_latency": (np.std(latency) / mean_latency) if mean_latency != 0 else 0,
    "cv_flight": (np.std(flight) / mean_flight) if mean_flight != 0 else 0,
}

print("\nFeature values:")
print(feat)

X_live = pd.DataFrame([feat])
X_live = X_live.reindex(columns=cols, fill_value=0)

pred = model.predict(X_live)[0]
proba = model.predict_proba(X_live)[0][1]

# shift confidence slightly
proba = (proba - 0.5) * 1.5 + 0.5
proba = max(0, min(1, proba))

if proba > 0.6:
    print("Prediction: Parkinson’s")
elif proba < 0.45:
    print("Prediction: Healthy")
else:
    print("Prediction: Uncertain")
    
print("Confidence:", proba)


Start typing for 20 seconds...

Feature values:
{'mean_hold': np.float64(0.18288997913661756), 'std_hold': np.float64(0.06156049804000599), 'mean_latency': np.float64(0.30107266338248), 'std_latency': np.float64(0.35383919665067703), 'mean_flight': np.float64(0.26943096866855376), 'std_flight': np.float64(0.5488021140180379), 'max_hold': np.float64(0.39827704429626465), 'min_hold': np.float64(0.060214996337890625), 'max_latency': np.float64(3.1317548751831055), 'min_latency': np.float64(0.02240729331970215), 'typing_speed': 5.467768134267252, 'pause_count': 5, 'cv_hold': np.float64(0.33659852951276636), 'cv_latency': np.float64(1.1752617878866103), 'cv_flight': np.float64(2.036893222520231)}


IndexError: index 1 is out of bounds for axis 0 with size 1

In [20]:
print("Train accuracy:", model.score(X_train, y_train))
print("Test accuracy:", model.score(X_test, y_test))

Train accuracy: 0.9086859688195991
Test accuracy: 0.7614678899082569
